Augustin, Swati

https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.preprocessing import LabelEncoder
from sklearn import preprocessing
from google.colab import drive
drive.mount ('/content/drive')
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")
#Imports all of the libraries and functions needed.



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Code to sort items as comma seperated values and helps to measure frequency of items
def items_to_text(items_set):
    return ", ".join(sorted(list(items_set)))


def support_of_itemset(cafe_items, items):
    if len(items) == 0:
        return 0.0
    return cafe_items[list(items)].all(axis=1).mean()

#sets our metrics for our association rules
min_support = 0.05
min_confidence = 0.45


In [ ]:

#Reads our csv for the data needed
df = pd.read_csv('/content/drive/MyDrive/IS-170/datasets/Lab_3_dataset/cafe_sales.csv')
#Clean up null values and replaces random error objects as null for easy clean up
df.drop('Transaction Date',axis=1,inplace=True)
df.drop('Transaction ID',axis=1,inplace=True)
df.drop('Total Spent',axis=1,inplace=True)
df.drop('Payment Method',axis=1,inplace=True)
df_r = df.replace('UNKNOWN', np.nan)
df_r = df.replace('ERROR', np.nan)
df_r.dropna(subset = ['Location','Quantity','Item'])
df_r.dropna(inplace=True)
print(df_r)

          Item Quantity Price Per Unit  Location
0       Coffee        2            2.0  Takeaway
1         Cake        4            3.0  In-store
2       Cookie        4            1.0  In-store
3        Salad        2            5.0   UNKNOWN
4       Coffee        2            2.0  In-store
...        ...      ...            ...       ...
9986  Sandwich        2            4.0  In-store
9991  Sandwich        3            4.0  Takeaway
9992  Smoothie        4            4.0  In-store
9995    Coffee        2            2.0   UNKNOWN
9999  Sandwich        3            4.0  In-store

[5572 rows x 4 columns]


In [ ]:
#Lists our values for the association rules to measure them
cafe_items = list(df_r.apply(lambda row: ','.join([row['Item'],row['Location']]), axis=1))
cafe_items = [item.split(',') for item in cafe_items]

In [ ]:
#Encoded our values to indetify the sections in boolean as 0 and 1
te = TransactionEncoder()
df_te = te.fit(cafe_items).transform(cafe_items)
cafe_items=pd.DataFrame(df_te, columns=te.columns_)
cafe_items = cafe_items.replace(False,0)
cafe_items = cafe_items.replace(True,1)
print(cafe_items)

      Cake  Coffee  Cookie  In-store  Juice  Salad  Sandwich  Smoothie  \
0        0       1       0         0      0      0         0         0   
1        1       0       0         1      0      0         0         0   
2        0       0       1         1      0      0         0         0   
3        0       0       0         0      0      1         0         0   
4        0       1       0         1      0      0         0         0   
...    ...     ...     ...       ...    ...    ...       ...       ...   
5567     0       0       0         1      0      0         1         0   
5568     0       0       0         0      0      0         1         0   
5569     0       0       0         1      0      0         0         1   
5570     0       1       0         0      0      0         0         0   
5571     0       0       0         1      0      0         1         0   

      Takeaway  Tea  UNKNOWN  
0            1    0        0  
1            0    0        0  
2            0    

/tmp/ipython-input-1100177649.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cafe_items = cafe_items.replace(True,1)


In [ ]:
 # Apriori frequent itemsets
    # This is where our items are identified by frequency and the rules used to measure based on lift or confidence
frequent_itemsets = apriori(cafe_items,min_support=min_support,use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
rules = rules.sort_values(["lift", "confidence"], ascending=False)

#Printing the transactions we have and what our metrics are looking for and then listing our top 15 items
print("DATASET")
print("Transactions:", len(cafe_items))
print("Unique items:", cafe_items.shape[1])
print()

print("APRIORI SETTINGS")
print("min_support   =", min_support)
print("min_confidence=", min_confidence)
print()

if frequent_itemsets.empty:
  print("No frequent itemsets found. Try lowering min_support.")


fi_print = frequent_itemsets.copy()
fi_print["itemsets"] = fi_print["itemsets"].apply(items_to_text)
print("FREQUENT ITEMSETS (top 15)")
print(fi_print.head(15).to_string(index=False))
print()


DATASET
Transactions: 5572
Unique items: 11

APRIORI SETTINGS
min_support   = 0.05
min_confidence= 0.45

FREQUENT ITEMSETS (top 15)
 support           itemsets
0.473259           In-store
0.473259           Takeaway
0.125987           Sandwich
0.125987              Salad
0.124551              Juice
0.120244             Coffee
0.119526             Cookie
0.118808               Cake
0.117373                Tea
0.112168           Smoothie
0.087222            UNKNOWN
0.062635 In-store, Sandwich
0.061917    In-store, Juice
0.061199    In-store, Salad
0.060481   Cookie, Takeaway



/usr/local/lib/python3.12/dist-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [ ]:
# Generate association rules
    # Here is how our table will visualize our metrics of our antecedents and consequents. We can see the values under suport,confidence,lift, and conviction
print("ASSOCIATION RULES (top 10)")
if rules.empty:
        print("No rules found. Try lowering min_support or min_confidence.")


rules_print = rules[["antecedents", "consequents", "support", "confidence", "lift", "conviction"]].copy()
rules_print["antecedents"] = rules_print["antecedents"].apply(items_to_text)
rules_print["consequents"] = rules_print["consequents"].apply(items_to_text)
rules_print["support"] = rules_print["support"].round(4)
rules_print["confidence"] = rules_print["confidence"].round(4)
rules_print["lift"] = rules_print["lift"].round(4)
rules_print["conviction"] = rules_print["conviction"].round(4)
print(rules_print.head(10).to_string(index=False))
print()

    # Prints out a easy to read description of our metrics
print("METRIC DEFINITIONS (for a rule A -> B)")
print("support(A)   = CI(A)")
print("support(A∪B) = CI(A and B)")
print("confidence   = CI(B|A) = support(A∪B) / support(A)")
print("lift         = confidence / support(B)")
print("conviction   = (1 - support(B)) / (1 - confidence)")
print()


ASSOCIATION RULES (top 10)
antecedents consequents  support  confidence   lift  conviction
     Cookie    Takeaway   0.0605      0.5060 1.0692      1.0663
   Sandwich    In-store   0.0626      0.4972 1.0505      1.0475
      Juice    In-store   0.0619      0.4971 1.0504      1.0474
   Smoothie    In-store   0.0547      0.4880 1.0311      1.0288
      Salad    In-store   0.0612      0.4858 1.0264      1.0243
     Coffee    Takeaway   0.0576      0.4791 1.0124      1.0112
        Tea    Takeaway   0.0558      0.4755 1.0048      1.0043
       Cake    Takeaway   0.0564      0.4743 1.0022      1.0020
      Salad    Takeaway   0.0587      0.4658 0.9843      0.9861
        Tea    In-store   0.0546      0.4648 0.9822      0.9843

METRIC DEFINITIONS (for a rule A -> B)
support(A)   = CI(A)
support(A∪B) = CI(A and B)
confidence   = CI(B|A) = support(A∪B) / support(A)
lift         = confidence / support(B)
conviction   = (1 - support(B)) / (1 - confidence)



In [ ]:
# Manual metric calculation demo for the top rule
top = rules.iloc[0]
A = top["antecedents"]
B = top["consequents"]

support_A = support_of_itemset(cafe_items, A)
support_B = support_of_itemset(cafe_items, B)
support_AB = support_of_itemset(cafe_items, set(A) | set(B))

confidence = 0.0 if support_A == 0 else support_AB / support_A
lift = np.nan if support_B == 0 else confidence / support_B
conviction = float("inf") if confidence == 1 else (1 - support_B) / (1 - confidence)


In [ ]:
#Prints an example of some of our finding and how it is measured. For example support A is Cookie, our antecendent. Now applying our lift rule we measure confidence as suport AUB over support A which looks like .0605/.1195 = .506
print("EXAMPLE (manual calculation for the top rule)")
print("A (antecedent):", items_to_text(A))
print("B (consequent):", items_to_text(B))
print("support(A)     =", round(support_A, 4))
print("support(B)     =", round(support_B, 4))
print("support(A∪B)   =", round(support_AB, 4))
print("confidence     = support(A∪B) / support(A)     =", round(confidence, 4))
print("lift           = confidence / support(B)      =", round(lift, 4))
if conviction == float("inf"):
      conviction_text = "inf"
else:
    conviction_text = str(round(conviction, 4))
print("conviction     = (1-support(B)) / (1-confidence)=", conviction_text)


EXAMPLE (manual calculation for the top rule)
A (antecedent): Cookie
B (consequent): Takeaway
support(A)     = 0.1195
support(B)     = 0.4733
support(A∪B)   = 0.0605
confidence     = support(A∪B) / support(A)     = 0.506
lift           = confidence / support(B)      = 1.0692
conviction     = (1-support(B)) / (1-confidence)= 1.0663


the Apriori association rule analysis on the café sales dataset identified meaningful patterns in customer purchasing behavior by discovering frequently bought item combinations. Using support, confidence, lift, and conviction, the analysis revealed strong product relationships that can support data-driven decisions such as bundling items, improving promotions, and optimizing store layout. Overall, the results demonstrate how association rule mining can provide valuable business insights from transactional data.